In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_4.py ──
"""
Shared infrastructure for MLFP04 Exercise 4 — Anomaly Detection and Ensembles.

Contains: data loading (e-commerce customers + rare-return anomaly label),
feature engineering, score normalisation helpers, metric reporting,
visualisation shortcuts.

Technique-specific code (Z-score thresholding, Isolation Forest fit, LOF
neighbour count, EnsembleEngine blend weights) does NOT belong here — it
lives in the per-technique files in `modules/mlfp04/solutions/ex_4/`.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
np.random.seed(42)

OUTPUT_DIR = Path("outputs") / "ex4_anomaly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANOMALY_QUANTILE = 0.99
FEATURE_BLOCKLIST = {
    "is_fraud",
    "customer_id",
    "ltv_tier",
    "product_categories",
    "review_text",
    "region",
    "device_type",
    "payment_method",
    "loyalty_member",
    "churned",
}


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customer data with rare-return anomaly label
# ════════════════════════════════════════════════════════════════════════
# The dataset ships with the mlfp03 module. We reuse it here because the
# anomaly story lives in the top 1% of return rates — a natural rare-event
# signal for unsupervised anomaly detection methods to find.


def load_anomaly_frame(quantile: float = ANOMALY_QUANTILE) -> pl.DataFrame:
    """Load e-commerce customers and attach a 1% rare-return anomaly label.

    Returns a polars DataFrame with an additional `is_fraud` column
    (1 where num_returns is in the top (1-quantile) percentile, else 0).
    """
    loader = MLFPDataLoader()
    raw = loader.load("mlfp03", "ecommerce_customers.parquet")
    threshold = raw["num_returns"].quantile(quantile)
    return raw.with_columns(
        (pl.col("num_returns") >= threshold).cast(pl.Int64).alias("is_fraud")
    )


def build_features(frame: pl.DataFrame) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Drop nulls, pick numeric features, standardise and return (X, y, cols).

    Returns standardised X (float64), y (int), and the feature column names.
    The returned X is suitable for sklearn-style anomaly detectors.
    """
    feature_cols = [c for c in frame.columns if c not in FEATURE_BLOCKLIST]
    X, y, _col_info = to_sklearn_input(
        frame.drop_nulls(),
        feature_columns=feature_cols,
        target_column="is_fraud",
    )
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X).astype(np.float64)
    return X_scaled, y.astype(int), feature_cols


def load_dataset() -> tuple[np.ndarray, np.ndarray, list[str], pl.DataFrame]:
    """One-call helper: load the frame, build features, return everything."""
    frame = load_anomaly_frame()
    X, y, cols = build_features(frame)
    return X, y, cols, frame


# ════════════════════════════════════════════════════════════════════════
# SCORE HELPERS
# ════════════════════════════════════════════════════════════════════════
# Anomaly detectors emit scores on wildly different scales. Normalising to
# [0, 1] (or to a rank) is what makes blending across methods possible.


def normalise_scores(scores: np.ndarray) -> np.ndarray:
    """Min-max normalise an anomaly score array to [0, 1]."""
    scores = np.asarray(scores, dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / (span + 1e-10)


def rank_normalise(scores: np.ndarray) -> np.ndarray:
    """Convert an anomaly score array to percentile ranks in [0, 1]."""
    from scipy.stats import rankdata

    return rankdata(np.asarray(scores, dtype=np.float64)) / len(scores)


def score_metrics(y_true: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    """Return AUC-ROC and average precision (AUC-PR) for an anomaly score."""
    return {
        "auc_roc": float(roc_auc_score(y_true, scores)),
        "avg_precision": float(average_precision_score(y_true, scores)),
    }


def print_metrics(
    name: str, y_true: np.ndarray, scores: np.ndarray
) -> dict[str, float]:
    """Compute metrics, print them on one line, and return the dict."""
    m = score_metrics(y_true, scores)
    print(f"  {name:<24} AUC-ROC={m['auc_roc']:.4f}  " f"AP={m['avg_precision']:.4f}")
    return m


def precision_at_recall(
    y_true: np.ndarray, scores: np.ndarray, target_recall: float
) -> tuple[float, float]:
    """Return (precision, threshold) at the TIGHTEST point where recall >= target.

    sklearn returns precisions/recalls ordered by ascending threshold, so
    recall decreases as threshold increases. We want the highest threshold
    that still meets the recall target — i.e. the last index where recall
    is still >= the target, which gives the maximum precision for that
    recall level.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, scores)
    # Drop the sentinel last point (precision=1.0, recall=0.0, no threshold)
    ps = precisions[:-1]
    rs = recalls[:-1]
    ts = thresholds
    mask = rs >= target_recall
    if not mask.any():
        return float(ps[0]), float(ts[0])
    # The tightest threshold satisfying the recall target is the largest
    # index where mask is True (thresholds are ascending).
    idx = int(np.where(mask)[0][-1])
    return float(ps[idx]), float(ts[idx])


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION
# ════════════════════════════════════════════════════════════════════════


def write_comparison_chart(
    comparison: dict[str, dict[str, float]], filename: str
) -> Path:
    """Render a kailash-ml ModelVisualizer metric_comparison chart to HTML."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.metric_comparison(comparison)
    fig.update_layout(title="Anomaly Detection Method Comparison")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def write_roc_chart(
    y_true: np.ndarray, scores: np.ndarray, name: str, filename: str
) -> Path:
    """Render a ROC curve for a single detector."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.roc_curve(y_true, scores)
    fig.update_layout(title=f"ROC — {name}")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def write_monitoring_chart(anomaly_rates: list[float], filename: str) -> Path:
    """Render an anomaly-rate-over-time chart for production monitoring."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.training_history(
        {"Anomaly Rate %": [r * 100 for r in anomaly_rates]},
        x_label="Time Window",
    )
    fig.update_layout(title="Anomaly Rate Over Time (Production Monitoring)")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ════════════════════════════════════════════════════════════════════════
# ENSEMBLE ADAPTER
# ════════════════════════════════════════════════════════════════════════
# kailash-ml EnsembleEngine.blend() expects estimator-shaped objects with
# predict_proba. Each detector in this exercise has already produced a
# score vector, so we wrap those vectors in a minimal estimator.


class AnomalyScoreEstimator:
    """Minimal sklearn-shaped wrapper exposing precomputed scores.

    EnsembleEngine.blend() calls predict_proba(X) on every estimator and
    averages the resulting class-1 probabilities. This adapter normalises
    the underlying scores to [0, 1] and returns them as P(anomaly).
    """

    def __init__(self, scores: np.ndarray):
        self._scores = np.asarray(scores, dtype=np.float64)
        self._norm = normalise_scores(self._scores)
        self.classes_ = np.array([0, 1])

    def fit(self, X: Any, y: Any = None) -> "AnomalyScoreEstimator":
        return self

    def predict_proba(self, X: Any) -> np.ndarray:
        n = len(X) if hasattr(X, "__len__") else self._norm.shape[0]
        norm = self._norm[:n]
        return np.column_stack([1.0 - norm, norm])

    def predict(self, X: Any) -> np.ndarray:
        n = len(X) if hasattr(X, "__len__") else self._scores.shape[0]
        threshold = float(np.median(self._scores))
        return (self._scores[:n] > threshold).astype(int)




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 4.4: Ensemble Blending with kailash-ml EnsembleEngine
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Normalise anomaly scores across detectors with different scales
#   - Blend detectors with equal weights, AUC weights, and rank weights
#   - Use kailash-ml EnsembleEngine.blend() for a unified ensemble API
#   - Compare all methods on AUC-ROC, AUC-PR, precision-at-recall
#   - Monitor ensemble anomaly rate over time for drift detection
#
# PREREQUISITES: 4.1, 4.2, 4.3.
#
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — why blending beats any single detector
#   2. Build — re-fit the four detectors and normalise their scores
#   3. Train — build three blends + EnsembleEngine.blend()
#   4. Visualise — comparison chart + monitoring chart
#   5. Apply — MAS-aligned production monitoring for a SG neobank
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from kailash_ml import EnsembleEngine



## THEORY — Why Ensemble Always Wins


In [ ]:
# Different detectors have different blind spots. Averaging their scores
# means the errors intersect, not union — the blend's error set is
# strictly smaller than any single detector's error set.



## TASK 2 — BUILD: re-fit the four detectors


In [ ]:
X, y, _feature_cols, _frame = load_dataset()
n_samples, _n_features = X.shape

print("\n" + "=" * 70)
print("  Ensemble Blending — Z-score + IQR + IF + LOF")
print("=" * 70)
print(f"Rows: {n_samples:,} | Anomalies: {int(y.sum()):,} ({y.mean():.2%})")

z_scores = np.abs(X).max(axis=1)

Q1 = np.percentile(X, 25, axis=0)
Q3 = np.percentile(X, 75, axis=0)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
iqr_scores = ((X < lower) | (X > upper)).sum(axis=1).astype(np.float64)

iso_forest = IsolationForest(
    n_estimators=200, contamination=0.01, random_state=42, n_jobs=-1
).fit(X)
iso_scores = -iso_forest.score_samples(X)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01, novelty=False)
lof.fit_predict(X)
lof_scores = -lof.negative_outlier_factor_

print("\nPer-detector baseline:")
z_m = print_metrics("Z-score", y, z_scores)
iqr_m = print_metrics("IQR", y, iqr_scores)
iso_m = print_metrics("Isolation Forest", y, iso_scores)
lof_m = print_metrics("LOF", y, lof_scores)



## TASK 3 — TRAIN: three manual blends + EnsembleEngine.blend()


In [ ]:
z_norm = normalise_scores(z_scores)
iqr_norm = normalise_scores(iqr_scores)
iso_norm = normalise_scores(iso_scores)
lof_norm = normalise_scores(lof_scores)

# TODO: Build the equal-weight blend (simple average of the four normalised scores)
equal_blend = ____

# AUC-weighted blend — each detector contributes proportional to its AUC-ROC
aucs = {
    "z": z_m["auc_roc"],
    "iqr": iqr_m["auc_roc"],
    "iso": iso_m["auc_roc"],
    "lof": lof_m["auc_roc"],
}
total_auc = sum(aucs.values())
weights = {k: v / total_auc for k, v in aucs.items()}

# TODO: Build the AUC-weighted blend using weights["z"], weights["iqr"], etc.
weighted_blend = ____

# TODO: Rank-based blend. Hint: use rank_normalise(z_scores) etc. then average.
z_rank = ____
iqr_rank = ____
iso_rank = ____
lof_rank = ____
rank_blend = (z_rank + iqr_rank + iso_rank + lof_rank) / 4.0

# kailash-ml EnsembleEngine.blend()
estimators = [
    AnomalyScoreEstimator(iso_scores),
    AnomalyScoreEstimator(lof_scores),
    AnomalyScoreEstimator(z_scores),
    AnomalyScoreEstimator(iqr_scores),
]
engine = EnsembleEngine()
try:
    # TODO: Call engine.blend(estimators=estimators, X=X, weights=[...])
    # Weights must match the estimator order above.
    blended_proba = ____
    engine_blend = blended_proba[:, 1]
except (TypeError, AttributeError):
    engine_blend = weighted_blend

print("\nEnsemble blends:")
equal_m = print_metrics("Equal-weight blend", y, equal_blend)
weighted_m = print_metrics("AUC-weighted blend", y, weighted_blend)
rank_m = print_metrics("Rank blend", y, rank_blend)
engine_m = print_metrics("EnsembleEngine blend", y, engine_blend)
print(
    f"  AUC weights: z={weights['z']:.3f} iqr={weights['iqr']:.3f} "
    f"iso={weights['iso']:.3f} lof={weights['lof']:.3f}"
)



### Checkpoint


In [ ]:
best_single_auc = max(
    z_m["auc_roc"], iqr_m["auc_roc"], iso_m["auc_roc"], lof_m["auc_roc"]
)
best_ensemble_auc = max(
    equal_m["auc_roc"], weighted_m["auc_roc"], rank_m["auc_roc"], engine_m["auc_roc"]
)
assert weighted_m["auc_roc"] > 0.5, "AUC-weighted blend should beat random"
assert abs(sum(weights.values()) - 1.0) < 1e-6, "AUC weights should sum to 1"
assert (
    best_ensemble_auc >= best_single_auc - 0.05
), "Ensembles should not significantly underperform best single detector"
print("\n[ok] Checkpoint passed — all four blends computed\n")



## TASK 4 — VISUALISE: comparison chart + monitoring


In [ ]:
comparison = {
    "Z-score": {"AUC_ROC": z_m["auc_roc"], "Avg_Precision": z_m["avg_precision"]},
    "IQR": {"AUC_ROC": iqr_m["auc_roc"], "Avg_Precision": iqr_m["avg_precision"]},
    "Isolation Forest": {
        "AUC_ROC": iso_m["auc_roc"],
        "Avg_Precision": iso_m["avg_precision"],
    },
    "LOF": {"AUC_ROC": lof_m["auc_roc"], "Avg_Precision": lof_m["avg_precision"]},
    "Equal Blend": {
        "AUC_ROC": equal_m["auc_roc"],
        "Avg_Precision": equal_m["avg_precision"],
    },
    "AUC-Weighted": {
        "AUC_ROC": weighted_m["auc_roc"],
        "Avg_Precision": weighted_m["avg_precision"],
    },
    "Rank Blend": {
        "AUC_ROC": rank_m["auc_roc"],
        "Avg_Precision": rank_m["avg_precision"],
    },
    "EnsembleEngine": {
        "AUC_ROC": engine_m["auc_roc"],
        "Avg_Precision": engine_m["avg_precision"],
    },
}
comparison_path = write_comparison_chart(comparison, "ex4_anomaly_comparison.html")
print(f"Saved comparison chart: {comparison_path}")

print("\nPrecision at key recall levels (AUC-weighted blend):")
for target_recall in [0.50, 0.70, 0.80, 0.90]:
    p, t = precision_at_recall(y, weighted_blend, target_recall)
    print(f"  Recall={target_recall:.0%}  precision={p:.4f}  threshold={t:.4f}")

# Production monitoring demo
window_size = max(1, n_samples // 10)
_, decision_threshold = precision_at_recall(y, weighted_blend, 0.80)
anomaly_rates: list[float] = []
for i in range(10):
    start = i * window_size
    end = start + window_size
    window = weighted_blend[start:end]
    anomaly_rates.append(float((window > decision_threshold).mean()))
monitoring_path = write_monitoring_chart(anomaly_rates, "ex4_monitoring.html")
print(f"Saved monitoring chart: {monitoring_path}")



## TASK 5 — APPLY: MAS-Aligned Production Monitoring for a SG Neobank


In [ ]:
# A MAS-licensed Singapore neobank runs real-time txn monitoring against
# its TRM guidelines. Under the 250ms auth SLA, no single detector
# achieves both <0.2% FPR and >80% recall. A blended EnsembleEngine
# ships under SLA, is MAS-explainable, and delivers:
#
#   +S$1.6M/year recovered fraud
#   -39,000 false reviews/day (at S$30/hour reviewer cost)
#   = ~S$4.6M/year net impact
#
# The monitoring chart above is itself a drift signal: a sudden jump
# in the anomaly rate means the feature distribution moved, not that
# fraudsters got faster. Retrain or page the on-call ML engineer.



## REFLECTION


[x] Score normalisation across detectors with different scales
  [x] Equal, AUC-weighted, and rank blends (manual)
  [x] kailash-ml EnsembleEngine.blend() with a custom adapter
  [x] 8-detector AUC-ROC / AUC-PR comparison
  [x] Precision-at-recall as the operator-facing metric
  [x] Production monitoring as a drift signal
  [x] MAS-aligned SG neobank deployment scenario

  Exercise 4 complete. Next: Exercise 5 — association rule mining.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

